# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. We will walk through loading the metadata, exploring available record sets and fields by their `@id`, extracting data for analysis, and performing simple exploratory data analysis and visualization.

### Dataset Source
The Croissant schema describing the dataset is available at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install the mlcroissant library if not already present
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset (metadata only at this point)
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
metadata = dataset.metadata  # This is a MetadataDocument object
print(f"{metadata.name}: {metadata.description}")
print(f"Citation: {metadata.citeAs}")

## 2. Data Overview
Review available record sets, their fields and column IDs.

**Note:** All record sets and fields are referenced by their `@id` as per Croissant specification.

In [ ]:
# List all available record sets
print("Record sets and their @id fields:")
record_set_ids = [rs['@id'] for rs in metadata._json_ld.get('recordSet', [])]
for rs in metadata._json_ld.get('recordSet', []):
    print(f"- {rs['@id']}: {rs.get('name', '(no name)')}")

# Display available fields within each record set
for rs in metadata._json_ld.get('recordSet', []):
    print(f"\nFields for record set {rs['@id']}: {rs.get('name', '')}")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"  - {field.get('@id', '')} ({field.get('name', '')}): type={field.get('dataType','')} column={field.get('column','')}")
        else:
            print(f"  - {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract data for all (or main) record sets discovered above
# If no record sets are returned by reading metadata, you may need to refer to the file's actual content via `dataset._json_ld['recordSet']`.

if not record_set_ids:
    # Try to extract from metadata._json_ld
    record_set_ids = [rs['@id'] for rs in metadata._json_ld.get('recordSet', [])]

if not record_set_ids:
    raise ValueError("No record sets found in dataset metadata.")

dataframes = dict()
# Try extracting first record set for demonstration (or loop over all if needed)
for rs_id in record_set_ids:
    print(f"Loading records for record set {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns in {rs_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No data for record set {rs_id}.")
    break  # For demonstration purposes, load only the first available record set

## 4. Exploratory Data Analysis (EDA)
We'll perform a few sample processing steps: filtering on a numeric field, normalization, and simple grouping.

In [ ]:
# Identify available numeric fields in the chosen DataFrame (first record set used above)
import numpy as np
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id]

# Guess likely numeric fields (such as 'MW', 'coverage_percent', etc.) and print types
print("\nData types in current record set DataFrame:")
print(df.dtypes)
potential_numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
if not potential_numeric_fields:
    # Try columns that look like numbers (e.g. forced conversion)
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass
    potential_numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()

print("\nNumeric fields detected:", potential_numeric_fields)

# Pick the first numeric field for EDA
if not potential_numeric_fields:
    raise Exception("No numeric fields found in DataFrame.")

numeric_field = potential_numeric_fields[0]
threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0

filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a likely categorical column if possible
categorical_fields = df.select_dtypes(include=['object']).columns.tolist()
group_field = categorical_fields[0] if categorical_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field. You can change the field names below as relevant.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We have loaded the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using `mlcroissant`, explored available record sets and their fields by `@id`, extracted tabular data, performed simple filtering/normalization/grouping, and visualized numeric distributions.

- The dataset structure and fields can be further explored using their `@id` (see section 2 above).
- You can adapt this notebook to any Croissant dataset by updating the URL and referencing record set/field IDs appropriately.
- For advanced analysis, consult the [mlcroissant documentation](https://mlcommons.org/working-groups/data/tooling/).
